### Utilities, preprocessing, dataset setup

In [ ]:
import os
import cv2
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from pycocotools.coco import COCO
import matplotlib.pyplot as plt
import random
import torch

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

IMAGE_SIZE = 1024


def gray_world_wb(img):
    imgf = img.astype(np.float32)
    mean = imgf.reshape(-1,3).mean(axis=0) + 1e-6
    scale = mean.mean() / mean
    out = np.clip(imgf * scale, 0, 255).astype(np.uint8)
    return out

def adaptive_gamma(img, target_v=0.5, clip=(0.7, 1.4)):
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    v = hsv[...,2].astype(np.float32)/255.0
    v_mean = float(np.clip(v.mean(), 0.05, 0.95))
    gamma = np.log(v_mean) / np.log(max(target_v, 1e-6))
    gamma = float(np.clip(gamma, clip[0], clip[1]))
    x = (img.astype(np.float32)/255.0) ** (1.0/gamma)
    return np.clip(x*255.0,0,255).astype(np.uint8)

def adaptive_clahe(img, base_clip=2.0, tile=(8,8)):
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    v = hsv[...,2]
    v_std = float(v.std())/255.0
    clip_limit = float(np.clip(base_clip + (0.8 - v_std)*1.0, 1.5, 3.0))
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile)
    hsv[...,2] = clahe.apply(v)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

def adaptive_color_deterministic(img):
    wb  = gray_world_wb(img)
    gam = adaptive_gamma(wb, target_v=0.5, clip=(0.8, 1.3))
    out = adaptive_clahe(gam, base_clip=2.0, tile=(8,8))
    return out


IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def get_val_test_transform_adaptive(IMAGE_SIZE=1024, norm="imagenet"):
    if norm == "imagenet":
        norm_tf = A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    elif norm == "minmax":
        norm_tf = A.Normalize(mean=(0,0,0), std=(1,1,1))
    elif norm == "none":
        norm_tf = A.Lambda(image=lambda x, **k: x)
    else:
        raise ValueError("norm must be 'imagenet' | 'minmax' | 'none'")

    return A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.Lambda(image=lambda x, **k: adaptive_color_deterministic(x)),
        norm_tf,
        ToTensorV2()
    ])

# Dataset class for loading COCO annotated images and masks
class COCOSegmentationDataset(Dataset):
    def __init__(self, img_dir, ann_path, transform):
        self.img_dir = img_dir
        self.coco = COCO(ann_path)
        self.image_ids = list(self.coco.imgs.keys())
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.img_dir, img_info['file_name'])

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = np.fliplr(image)  # Match training flips

        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        mask = np.zeros((img_info['height'], img_info['width']), dtype=np.uint8)
        for ann in anns:
            mask = np.maximum(mask, self.coco.annToMask(ann))
        mask = np.fliplr(mask)

        if mask.shape[:2] != image.shape[:2]:
            mask = cv2.resize(mask, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)

        augmented = self.transform(image=image, mask=mask)
        image = augmented['image']
        mask = (augmented['mask'] > 0).unsqueeze(0).float()
        return image, mask


# Create test dataset
test_img_dir = "/Users/snirtahasa/Almond_Research/Training/Notebooks/Almons-Trees-8/test"
test_ann_path = os.path.join(test_img_dir, "_annotations.coco.json")
test_transform = get_val_test_transform_adaptive(IMAGE_SIZE=IMAGE_SIZE, norm="imagenet")
test_dataset = COCOSegmentationDataset(test_img_dir, test_ann_path, test_transform)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)

# Optional visualization of one batch
def show_batch(images, masks):
    for i in range(len(images)):
        img = images[i].permute(1, 2, 0).cpu().numpy()
        mask = masks[i][0].cpu().numpy()

        plt.figure(figsize=(10, 5))
        plt.subplot(1, 2, 1)
        plt.imshow(img)
        plt.title("Image")
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.imshow(mask, cmap='gray')
        plt.title("Mask")
        plt.axis('off')
        plt.show()

for images, masks in test_loader:
    show_batch(images, masks)
    break


### Model loading, adaptive splitting and visualization

In [ ]:
import torch
import numpy as np
from scipy import ndimage as ndi
from skimage import morphology, measure, segmentation as _seg
from skimage.feature import canny
from skimage.morphology import disk, thin
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt


device = torch.device("mps" if torch.backends.mps.is_available() else
                      "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load your model matching encoder architecture and pretrained weights settings
model_ImNet_Plus = smp.UnetPlusPlus(
    encoder_name="efficientnet-b3",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None,
).to(device)

def load_best_model_checkpoint(model, checkpoint_path="best_model.pth", device=device):
    ckpt = torch.load(checkpoint_path, map_location=device)
    state_dict = ckpt.get('model_state_dict', ckpt)
    missing, unexpected = model.load_state_dict(state_dict, strict=True)
    if missing or unexpected:
        print("⚠️ load_state_dict mismatches")
        if missing:
            print("  Missing keys:", missing)
        if unexpected:
            print("  Unexpected keys:", unexpected)
    model.eval()
    print("Loaded best model checkpoint.")
    return model

model_ImNet_Plus = load_best_model_checkpoint(model_ImNet_Plus)

# Load best model checkpoint from current directory
def load_best_model_checkpoint(model, checkpoint_path="best_model.pth", device=device):
    ckpt = torch.load(checkpoint_path, map_location=device)
    state_dict = ckpt.get('model_state_dict', ckpt)
    missing, unexpected = model.load_state_dict(state_dict, strict=True)
    if missing or unexpected:
        print("⚠️ load_state_dict mismatches")
        if missing:
            print("  Missing keys:", missing)
        if unexpected:
            print("  Unexpected keys:", unexpected)
    model.to(device)
    model.eval()
    print("Loaded best model checkpoint.")
    return model

model_ImNet_Plus = load_best_model_checkpoint(model_ImNet_Plus)

# Adaptive instance splitting with postprocessing
def adaptive_instance_split(
    pred_mask,
    min_area=220,
    max_area=4500,
    open_radius=1,
    cut_px=1.5,
    valley_smooth=0.02,
    bins_min=48,
    bins_max=256,
    step_deg=10,
    jump_factor=1.4,
    radius_q=(0.33, 0.63),
    spacing_scale=0.96,
    angle_jitter_deg=2.0,
    thin_cuts=True,
):
    def _radii_from_centroid(reg_sub, cy, cx, step_deg=15, max_step=None):
        Hs, Ws = reg_sub.shape
        if max_step is None:
            max_step = np.hypot(Hs, Ws)
        deg = np.arange(0, 180, step_deg, dtype=np.float32)
        ang = np.deg2rad(deg)
        radii = np.zeros_like(ang, dtype=np.float32)
        for i, a in enumerate(ang):
            dx, dy = np.cos(a), np.sin(a)
            x, y = float(cx), float(cy)
            r = 0.0
            while 0 <= int(round(y)) < Hs and 0 <= int(round(x)) < Ws and reg_sub[int(round(y)), int(round(x))]:
                x += dx; y += dy; r += 1.0
                if r > max_step:
                    break
            radii[i] = r
        return deg, ang, radii

    def _valleys_1d(profile, k_cuts, smooth_sigma):
        if k_cuts <= 0 or profile.size < 8:
            return []
        p = ndi.gaussian_filter1d(profile.astype(np.float32), smooth_sigma, mode='nearest')
        inv = -p
        is_peak = (inv > np.r_[inv[1:], -np.inf]) & (inv > np.r_[-np.inf, inv[:-1]])
        idx = np.where(is_peak)[0]
        if idx.size == 0:
            return []
        order = np.argsort(p[idx])
        idx = idx[order[:k_cuts]]
        idx.sort()
        return idx.tolist()

    def _half(span, cut_px, band_frac=0.1, min_cut_px=1.0):
        return max(min_cut_px/2.0, float(cut_px)/2.0)

    # Preprocessing mask
    m = pred_mask.astype(bool)
    if open_radius > 0:
        m = morphology.opening(m, disk(int(open_radius)))
    m = morphology.remove_small_objects(m, min_size=int(min_area))
    if not m.any():
        return np.zeros_like(pred_mask, np.int32)

    labeled = measure.label(m, connectivity=1)

    base_radii = []
    obj_info = []
    jump_angles = []

    for r in measure.regionprops(labeled):
        if r.area < min_area:
            continue
        minr, minc, maxr, maxc = r.bbox
        reg_sub = (labeled[minr:maxr, minc:maxc] == r.label)
        cy, cx = r.centroid; cy -= minr; cx -= minc

        deg, ang, radii = _radii_from_centroid(reg_sub, cy, cx, step_deg=step_deg)
        lo, hi = np.quantile(radii, radius_q)
        base = 0.5*(lo + hi)
        base_radii.append(base)

        jump_idx = np.where(radii > (jump_factor * base))[0]
        if jump_idx.size:
            jump_angles.extend(deg[jump_idx].tolist())

        obj_info.append((r.label, (minr, minc, maxr, maxc), (cy, cx), deg, ang, radii))

    R_est = float(np.median(base_radii)) if base_radii else 3.0
    R_est = max(1.0, R_est)

    if len(jump_angles) >= 3:
        hist, edges = np.histogram(jump_angles, bins=36, range=(0,180))
        peak_deg = float(0.5*(edges[np.argmax(hist)] + edges[np.argmax(hist)+1]))
        ang_row_base = np.deg2rad(peak_deg)
    else:
        bg = (~m).astype(np.float32)
        g  = ndi.gaussian_filter(bg, 2.0)
        gy, gx = np.gradient(g)
        Jxx = ndi.gaussian_filter(gx*gx, 2.0)
        Jxy = ndi.gaussian_filter(gx*gy, 2.0)
        Jyy = ndi.gaussian_filter(gy*gy, 2.0)
        theta = 0.5*np.arctan2(2*Jxy, (Jxx - Jyy + 1e-8))
        vals = theta[bg > np.percentile(bg, 50)]
        ang_row_base = float(np.median(vals)) if vals.size else 0.0

    jitter_deg = float(angle_jitter_deg)
    jitter_list = [-np.deg2rad(jitter_deg), 0.0, +np.deg2rad(jitter_deg)] if jitter_deg > 0 else [0.0]

    cut_global = np.zeros_like(m, bool)

    for jitter in jitter_list:
        ang_row = (ang_row_base + jitter) % np.pi
        ang_col = (ang_row + np.pi/2.0) % np.pi
        cos_r, sin_r = np.cos(ang_row), np.sin(ang_row)
        cos_c, sin_c = np.cos(ang_col), np.sin(ang_col)

        for (lbl, bbox, (cy, cx), deg, ang, radii) in obj_info:
            minr, minc, maxr, maxc = bbox
            reg_sub = (labeled[minr:maxr, minc:maxc] == lbl)
            Hs, Ws = reg_sub.shape

            yy, xx = np.mgrid[0:Hs, 0:Ws]
            yy_abs, xx_abs = yy + minr, xx + minc

            t_row = xx_abs * cos_r + yy_abs * sin_r
            s_col = -xx_abs * sin_r + yy_abs * cos_r
            t_vals = t_row[reg_sub]; s_vals = s_col[reg_sub]
            t_min, t_max = float(t_vals.min()), float(t_vals.max())
            s_min, s_max = float(s_vals.min()), float(s_vals.max())
            Lr = t_max - t_min; Wc = s_max - s_min + 1e-6

            t_col = xx_abs * cos_c + yy_abs * sin_c
            s_row = -xx_abs * sin_c + yy_abs * cos_c
            tc_vals = t_col[reg_sub]; sr_vals = s_row[reg_sub]
            tc_min, tc_max = float(tc_vals.min()), float(tc_vals.max())
            sr_min, sr_max = float(sr_vals.min()), float(sr_vals.max())
            Lc = tc_max - tc_min; Wr = sr_max - sr_min + 1e-6

            D = 2.0 * R_est * float(spacing_scale)
            n_row = int(np.round(Lr / max(D, 1.0)))
            n_col = int(np.round(Lc / max(D, 1.0)))

            cut_local = np.zeros_like(reg_sub, bool)

            if n_row >= 2 and (Lr / Wc) >= 1.2:
                nb = int(np.clip(np.round(Lr), bins_min, bins_max))
                idx = np.clip(((t_row - t_min) / (Lr + 1e-6) * nb).astype(int), 0, nb-1)
                prof = np.bincount(idx[reg_sub], minlength=nb)
                k = max(1, n_row - 1)
                valleys = _valleys_1d(prof, k, smooth_sigma=max(1.0, valley_smooth*nb))
                if valleys:
                    t_bounds = [t_min + (vi + 0.5) * (Lr / nb) for vi in valleys]
                    half = _half(Wc, cut_px)
                    for tb in t_bounds:
                        cut_local |= (np.abs(t_row - tb) <= half)

            if n_col >= 2 and (Lc / Wr) >= 1.2:
                nb = int(np.clip(np.round(Lc), bins_min, bins_max))
                idx = np.clip(((t_col - tc_min) / (Lc + 1e-6) * nb).astype(int), 0, nb-1)
                prof = np.bincount(idx[reg_sub], minlength=nb)
                k = max(1, n_col - 1)
                valleys = _valleys_1d(prof, k, smooth_sigma=max(1.0, valley_smooth*nb))
                if valleys:
                    t_bounds = [tc_min + (vi + 0.5) * (Lc / nb) for vi in valleys]
                    half = _half(Wr, cut_px)
                    for tb in t_bounds:
                        cut_local |= (np.abs(t_col - tb) <= half)

            if thin_cuts and cut_local.any():
                cut_local = thin(cut_local)
                if cut_px > 1.0:
                    rad = int(max(0, np.round(cut_px/2.0) - 1))
                    if rad > 0:
                        cut_local = morphology.binary_dilation(cut_local, disk(rad))

            cut_global[minr:maxr, minc:maxc] |= (cut_local & reg_sub)

    if cut_global.any():
        m = m & (~cut_global)
    m = morphology.remove_small_objects(m, min_size=int(min_area))
    labeled = measure.label(m, connectivity=1)

    # Postprocess large merged crowns by canny edge and connected components
    final_labels = np.zeros_like(labeled)
    label_counter = 1
    for region in measure.regionprops(labeled):
        if region.area > max_area:
            sub_mask = (labeled == region.label)
            edges = canny(sub_mask.astype(np.uint8), sigma=1.0)
            markers = ndi.label(~edges & sub_mask)[0]
            for sub_r in measure.regionprops(markers):
                if sub_r.area >= min_area:
                    coords = sub_r.coords
                    final_labels[coords[:,0], coords[:,1]] = label_counter
                    label_counter += 1
        else:
            coords = region.coords
            final_labels[coords[:,0], coords[:,1]] = label_counter
            label_counter += 1

    # Filter small or irregular instances
    for region in measure.regionprops(final_labels):
        if region.area < min_area // 2 or region.solidity < 0.8:
            coords = region.coords
            final_labels[coords[:,0], coords[:,1]] = 0

    final_labels = measure.label(final_labels > 0, connectivity=1)
    return final_labels

# Visualization helper
def overlay_instances_on_image(image_rgb_uint8, labels, alpha=0.35):
    from skimage import color
    img_f = np.clip(image_rgb_uint8.astype(np.float32) / 255.0, 0, 1)
    vis = color.label2rgb(labels, image=img_f, bg_label=0, alpha=float(alpha))
    return (np.clip(vis, 0, 1) * 255).astype(np.uint8)

def denormalize_imagenet(tensor_img, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    if isinstance(tensor_img, torch.Tensor):
        tensor_img = tensor_img.detach().cpu()
    img = tensor_img.permute(1, 2, 0).numpy()
    img = img * np.array(std)[None, None, :] + np.array(mean)[None, None, :]
    img = np.clip(img, 0, 1)
    return (img * 255).astype(np.uint8)

# Visualization of your results using adaptive splitting
def visualize_predictions_with_adaptive_split(
    model, dataset, device, max_samples=10, thresh=0.66,
):
    import torch
    model.eval()
    n = min(len(dataset), max_samples)
    for i in range(n):
        image_t, mask_t = dataset[i]
        image_b = image_t.unsqueeze(0).to(device)
        with torch.no_grad():
            logits = model(image_b)
        prob = torch.sigmoid(logits)[0, 0].cpu().numpy()
        pred_mask = (prob > thresh).astype(np.uint8)
        img_vis = denormalize_imagenet(image_t)

        labels = adaptive_instance_split(pred_mask)
        n_instances = int(labels.max())
        gt_mask = mask_t.squeeze(0).cpu().numpy().astype(np.uint8)

        sem_overlay = img_vis.copy()
        sem_overlay[pred_mask.astype(bool)] = (
            0.5 * sem_overlay[pred_mask.astype(bool)] + 0.5 * np.array([0, 255, 0], dtype=np.float32)
        ).astype(np.uint8)

        inst_overlay = overlay_instances_on_image(img_vis, labels, alpha=0.35)
        boundaries = _seg.find_boundaries(labels, mode="outer")
        boundary_overlay = img_vis.copy()
        boundary_overlay[boundaries] = [255, 0, 0]

        fig, axs = plt.subplots(1, 5, figsize=(20, 4))
        axs[0].imshow(img_vis);              axs[0].set_title("Image");            axs[0].axis("off")
        axs[1].imshow(gt_mask, cmap="gray"); axs[1].set_title("Ground Truth");    axs[1].axis("off")
        axs[2].imshow(sem_overlay);          axs[2].set_title("Semantic Overlay"); axs[2].axis("off")
        axs[3].imshow(inst_overlay);         axs[3].set_title(f"Instances (N={n_instances})"); axs[3].axis("off")
        axs[4].imshow(boundary_overlay);     axs[4].set_title("Instance Boundaries"); axs[4].axis("off")
        plt.tight_layout()
        plt.show()

        print(f"[{i+1}/{n}] separated instances: {n_instances}")

# Example usage (ensure test_dataset is defined in your environment):
visualize_predictions_with_adaptive_split(model_ImNet_Plus, test_dataset, device, max_samples=10, thresh=0.66)


### Prediction and visualization on folder images with padding approach (aspect-ratio preserved)

In [ ]:
import os
import cv2
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch
import matplotlib.pyplot as plt
from skimage import segmentation as _seg

def preprocess_image_with_padding(image_path, target_size=IMAGE_SIZE, use_adaptive_color=True):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if use_adaptive_color:
        img = adaptive_color_deterministic(img)

    h, w = img.shape[:2]
    scale = target_size / max(h, w)
    nh, nw = int(h * scale), int(w * scale)

    img_resized = cv2.resize(img, (nw, nh))

    top = (target_size - nh) // 2
    bottom = target_size - nh - top
    left = (target_size - nw) // 2
    right = target_size - nw - left

    img_padded = cv2.copyMakeBorder(img_resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(0,0,0))

    transform = A.Compose([
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2()
    ])

    augmented = transform(image=img_padded)
    return augmented['image'].unsqueeze(0), (top, bottom, left, right), (nh, nw)


def predict_folder_with_padding(image_folder, model, device, max_images=10, thresh=0.66):
    image_files = sorted([f for f in os.listdir(image_folder) if f.lower().endswith(('.jpg','.png','.jpeg','.tif'))])[:max_images]
    print(f"Processing {len(image_files)} images from {image_folder}")
    for i, fname in enumerate(image_files):
        image_path = os.path.join(image_folder, fname)
        print(f"[{i+1}/{len(image_files)}] Processing {fname}")

        # Preprocess image
        img_tensor, pad, resized_shape = preprocess_image_with_padding(image_path, target_size=IMAGE_SIZE, use_adaptive_color=True)


        # Visual check of the preprocessing output: denormalize and display
        import matplotlib.pyplot as plt
        img_np = img_tensor[0].permute(1,2,0).cpu().numpy()
        mean = np.array(IMAGENET_MEAN)
        std = np.array(IMAGENET_STD)
        img_vis = img_np * std + mean
        img_vis = (img_vis * 255).clip(0,255).astype(np.uint8)
        plt.figure(figsize=(6,6))
        plt.title(f"Preprocessed image preview: {fname}")
        plt.imshow(img_vis)
        plt.axis('off')
        plt.show()

        # Proceed with prediction and postprocessing
        labels, prob_mask = predict_and_postprocess_unpad(img_tensor, model, device, pad, resized_shape, thresh)

        visualize_prediction(image_path, labels, prob_mask, thresh=thresh)


# Example usage:
predict_folder_with_padding("/Users/snirtahasa/Desktop/Almonds - Project/mosaic", model_ImNet_Plus, device, max_images=10, thresh=0.66)


In [ ]:
import os
import cv2
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch
import matplotlib.pyplot as plt
from skimage.exposure import match_histograms

def get_reference_image_from_folder(folder_path):
    # Pick first image file in folder as reference
    for ext in ['.jpg', '.jpeg', '.png', '.tif', '.bmp']:
        files = [f for f in os.listdir(folder_path) if f.lower().endswith(ext)]
        if len(files) > 0:
            ref_path = os.path.join(folder_path, files[0])
            ref_img = cv2.imread(ref_path)
            ref_img = cv2.cvtColor(ref_img, cv2.COLOR_BGR2RGB)
            print(f"Using reference image for histogram matching: {files[0]}")
            return ref_img
    raise FileNotFoundError(f"No reference image found in folder {folder_path}")

def preprocess_image_with_histmatch(image_path, reference_img, target_size=IMAGE_SIZE, use_adaptive_color=True):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    img_matched = match_histograms(img, reference_img, channel_axis=2)
    img_matched = np.clip(img_matched, 0, 255).astype(np.uint8)

    if use_adaptive_color:
        img_matched = adaptive_color_deterministic(img_matched)

    h, w = img_matched.shape[:2]
    scale = target_size / max(h, w)
    nh, nw = int(h * scale), int(w * scale)
    img_resized = cv2.resize(img_matched, (nw, nh))
    top = (target_size - nh) // 2
    bottom = target_size - nh - top
    left = (target_size - nw) // 2
    right = target_size - nw - left
    img_padded = cv2.copyMakeBorder(img_resized, top, bottom, left, right,
                                    cv2.BORDER_CONSTANT, value=(0,0,0))

    transform = A.Compose([
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2()
    ])

    augmented = transform(image=img_padded)
    return augmented['image'].unsqueeze(0), (top, bottom, left, right), (nh, nw)

def predict_folder_with_histmatch_ref(image_folder, model, device,
                                      max_images=10, thresh=0.66, use_adaptive_color=True):
    reference_img = get_reference_image_from_folder(image_folder)

    image_files = sorted([f for f in os.listdir(image_folder) if f.lower().endswith(('.jpg','.png','.jpeg','.tif'))])[:max_images]
    print(f"Processing {len(image_files)} images from {image_folder}")

    for i, fname in enumerate(image_files):
        image_path = os.path.join(image_folder, fname)
        print(f"[{i+1}/{len(image_files)}] Processing {fname}")

        img_tensor, pad, resized_shape = preprocess_image_with_histmatch(image_path, reference_img,
                                                                        target_size=IMAGE_SIZE,
                                                                        use_adaptive_color=use_adaptive_color)

        img_np = img_tensor[0].permute(1,2,0).cpu().numpy()
        mean = np.array(IMAGENET_MEAN)
        std = np.array(IMAGENET_STD)
        img_vis = img_np * std + mean
        img_vis = (img_vis * 255).clip(0,255).astype(np.uint8)
        plt.figure(figsize=(6,6))
        plt.title(f"Preprocessed image preview: {fname}")
        plt.imshow(img_vis)
        plt.axis('off')
        plt.show()

        labels, prob_mask = predict_and_postprocess_unpad(img_tensor, model, device, pad, resized_shape, thresh)

        visualize_prediction(image_path, labels, prob_mask, thresh=thresh)

# Example usage:
predict_folder_with_histmatch_ref(
     "/Users/snirtahasa/Desktop/Almonds - Project/mosaic",
     model_ImNet_Plus,
     device,
     max_images=10,
     thresh=0.66,
     use_adaptive_color=True
 )
